# JSON Data Overview

This JSON record provides detailed information about a **contact's booking history**. The main purpose is to store and manage data related to bookings, customers, and associated services such as accommodation, courses, payments, and additional services.

## Top-Level Keys
1. **`contact_id`**: A unique identifier for the contact (e.g., 5689).
2. **`bookings`**: Contains all booking-related data:
   - `hits`: Number of bookings (e.g., 1).
   - `entries`: A nested structure for individual booking details.

---

## `entries` Structure (Booking Details)
Each booking is represented as an entry containing the following key categories:

### General Booking Information
- **`school_id`**: ID of the school where the booking was made.
- **`id`**: Unique ID of the booking.
- **`type`**: Type of record (e.g., `booking`).
- **`inbox`**: Inbox type (e.g., `default`).
- **`document_number_all`**: List of all document numbers associated with the booking.
- **`invoice_status`**: Status of invoices (e.g., `invoice_created`, `invoice_outdated`, `not_cancelled`).
- **`course_name_en`**: Name of the course (e.g., "Evening Classes").
- **`status`**: Current status of the booking (e.g., `ready`).

---

### Customer Information
- **`customer_id`**: Customer identifier.
- **`customer_name`**: Full name of the customer.
- **`customer_firstname`** / **`customer_lastname`**: First and last names of the customer.
- **`email`**: Customer email address.
- **`customer_gender`**: Gender of the customer (e.g., `2` for female).
- **`customer_birthday`**: Date of birth (e.g., `1988-03-14`).
- **`customer_age_course_start`**: Age of the customer at the start of the course.

---

### Course Details
- **`course_ids`**: ID of the course.
- **`course_name_en`**: English name of the course.
- **`course_language_en`**: Language of the course.
- **`course_period`**: Start and end dates for the course in timestamp format.
- **`course_weeks`**: Total number of course weeks.
- **`current_level_intern_full`**: Current level of the student (e.g., "Upper Intermediate (B2)").

---

### Accommodation Details
- **`accommodation_category`**: Type of accommodation.
- **`accommodation_from`** / **`accommodation_until`**: Start and end dates for accommodation.
- **`accommodation_address`**: Details of the accommodation address, including:
  - **Street**
  - **City**
  - **ZIP Code**
- **`accommodation_provider_confirmed`**: Whether the accommodation provider confirmed the booking.

---

### Payment Details
- **`amount_total_original`**: Total amount for the booking (e.g., 420).
- **`amount_payed_original`**: Amount paid (e.g., 420).
- **`amount_open`**: Outstanding payment (e.g., 0).
- **`paymentterms_info`**: Details of payment terms.

---

### Emergency Contact
- **`emergency_name`**: Name of the emergency contact.
- **`emergency_phone`**: Phone number of the emergency contact.

---

### Additional Details
- **`created_original`**: Creation timestamp for the booking.
- **`editor_id`**: ID of the last editor.
- **`changed_original`**: Timestamp of the last modification.
- **`converted`**: Indicates if the booking has been converted.

---

## Purpose
This JSON record is designed to:
- Store comprehensive information about bookings, payments, and services.
- Provide a detailed view of customer profiles, course details, and accommodation data.
- Enable tracking and auditing of booking statuses and changes over time.

## Usage
- Suitable for database storage and reporting.
- Useful for analytics on booking trends, payment history, and customer demographics.


In [ ]:
import requests
import json
import os
import time

# Replace with your actual API URL and Token
BASE_URL = "https://your-school.fidelo.com/api/1.0/ts/bookings"
API_TOKEN = os.environ["API_TOKEN"]  # Replace with your actual Bearer Token

# Define Filters
pp_status = 0
changed_original_start = "2024-12-31"
changed_original_end = "2025-01-02"

# Headers for the request
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# Paths for output files
output_dir = "./"
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exQists

bookings_file_path = os.path.join(output_dir, f"booking_list_update{changed_original_start}_{changed_original_end}.json")

# Containers for storing fetched data
booking_list = []  # Full booking details

url = f"{BASE_URL}?profile_pictures={pp_status}&filter[changed_original]={changed_original_start}%2C{changed_original_end}"

try:
    # Start timestamp
    call_timestamp_start = time.time()

    # Send the GET request
    response = requests.get(url, headers=headers)

    # End timestamp
    call_timestamp_end = time.time()

    # Calculate elapsed time
    elapse_sec = round(call_timestamp_end - call_timestamp_start, 2)

    # Parse response status and JSON data
    response_status = response.status_code
    data = response.json() if response_status == 200 else {}

    # Extract 'hits' from the data (if available)
    hits = data.get("hits", 0)

    # Calculate hit_by_elapse
    hit_by_elapse = round(elapse_sec / hits, 2) if hits > 0 else None

    # Construct the output structure under the "root" key
    output = {
            "status": response_status,
            "pp_status": pp_status,
            "changed_original_start": changed_original_start,
            "changed_original_end": changed_original_end,
            "call_timestamp_start": call_timestamp_start,
            "call_timestamp_end": call_timestamp_end,
            "elapse_sec": elapse_sec,
            "hit_by_elapse": hit_by_elapse,
            "hits": hits,
            "bookings": data.get("entries", {})
    }

    # Save the output to a file
    with open(bookings_file_path, "w") as file:
        json.dump(output, file, indent=4)

    print(f"Data successfully saved to {bookings_file_path}.")

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")